## MODELING
#### Selección de features

Partimos de las siguientes variables/columnas:
- metros
- baños_limpio
- habitaciones_limpio
- zona
- tipo_inmueble
- ascensor_limpio
- localizacion_limpio
- flag_rebaja
- flag_loft
- flag_nuda_propiedad
- flag_proindiviso
- flag_subasta
- flag_okupada
- flag_alquilada

Candidatas a ser evaluadas más adelante con CatBoost:
- barrio
- planta_limpio

In [1]:
import pandas as pd
import numpy as np

In [2]:
target = "PrecioActual"

numericas = ["metros", "baños_limpio", "habitaciones_limpio"]
categoricas = ["zona", "tipo_inmueble", "ascensor_limpio", "localizacion_limpio"]
binarias = ["flag_rebaja", "flag_loft", "flag_nuda_propiedad", "flag_proindiviso", "flag_subasta", "flag_okupada", "flag_alquilada"]

# Las candidatas para el Catboost cuando toque
extras = ["barrio", "planta_limpio"]

features = numericas + categoricas + binarias
features

['metros',
 'baños_limpio',
 'habitaciones_limpio',
 'zona',
 'tipo_inmueble',
 'ascensor_limpio',
 'localizacion_limpio',
 'flag_rebaja',
 'flag_loft',
 'flag_nuda_propiedad',
 'flag_proindiviso',
 'flag_subasta',
 'flag_okupada',
 'flag_alquilada']

In [3]:
train = pd.read_csv("../data_sample/train.csv")

X_train = train[features]
y_train = train[target]

X_train.shape

(8947, 14)

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Resumen rápido de cómo actuar con las Pipelines y por qué hacerlo así con un ColumnTransformer (no se podría de otra manera tampoco)
# No puedes calcular la mediana de un texto. Por eso el ColumnTransformer. Le indicas en una tupla de ("nombre", qué, a qué) y eliges lo que quieres hacer
# En este caso: A las numéricas: pipeline, imputar mediana, escalar
# a las categóricas: one hot
# a las binarias: nada, están ya hechas
preprocessing = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),]), numericas),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas),
    ("bin", "passthrough", binarias)
])

In [5]:
from sklearn.linear_model import LinearRegression

reg_lin = Pipeline([("prep", preprocessing),
                    ("reg", LinearRegression())])
reg_lin

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('reg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the outp

In [6]:
from sklearn.model_selection import cross_validate, KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)

resultado = cross_validate(reg_lin, X_train, y_train, cv=cv, scoring=["neg_root_mean_squared_error", "neg_mean_absolute_error", "r2"])

In [7]:
print("RMSE:", -resultado["test_neg_root_mean_squared_error"].mean())
print("MAE :", -resultado["test_neg_mean_absolute_error"].mean())
print("R2  :", resultado["test_r2"].mean())

RMSE: 717943.0574746515
MAE : 356861.28000447544
R2  : 0.6675770940721751


#### Baseline Linear Regression
### MAE: -356861.28000447544

## XGBoost:


In [8]:
from xgboost import XGBRegressor

xgboost = Pipeline([("prep", preprocessing),
                    ("xgboost", XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=7, random_state=42, n_jobs=-1))])
xgboost

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('xgboost', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the 

In [9]:
from sklearn.compose import TransformedTargetRegressor

xgboost_transformed_target = TransformedTargetRegressor(regressor=xgboost, func=np.log1p, inverse_func=np.expm1)

In [10]:
resultado = cross_validate(xgboost_transformed_target, X_train, y_train, cv=cv, scoring=["neg_root_mean_squared_error", "neg_mean_absolute_error", "r2"])

In [11]:
print("RMSE:", -resultado["test_neg_root_mean_squared_error"].mean())
print("MAE :", -resultado["test_neg_mean_absolute_error"].mean())
print("R2  :", resultado["test_r2"].mean())

RMSE: 522246.89375
MAE : 220515.5375
R2  : 0.8235477566719055


### XGBOOST
### MAE: 220515.5375

## LIGHTGBM

In [12]:
from lightgbm import LGBMRegressor

lightgbm = Pipeline([("prep", preprocessing),
                     ("lightgbm", LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=31, max_depth=-1, min_child_samples=20, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1 ))])

lightgbm

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('lightgbm', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the

In [13]:
lightgbm_transformed_target = TransformedTargetRegressor(regressor=lightgbm, func=np.log1p, inverse_func=np.expm1)

resultado = cross_validate(lightgbm_transformed_target, X_train, y_train, cv=cv, scoring=["neg_root_mean_squared_error", "neg_mean_absolute_error", "r2"])

print("RMSE:", -resultado["test_neg_root_mean_squared_error"].mean())
print("MAE :", -resultado["test_neg_mean_absolute_error"].mean())
print("R2  :", resultado["test_r2"].mean())

c:\Users\ramir\Documents\GitHub\marzo\ML-idealista\ML-idealista\.venv ML-Idealista\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\ramir\Documents\GitHub\marzo\ML-idealista\ML-idealista\.venv ML-Idealista\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\ramir\Documents\GitHub\marzo\ML-idealista\ML-idealista\.venv ML-Idealista\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\ramir\Documents\GitHub\marzo\ML-idealista\ML-idealista\.venv ML-Idealista\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


RMSE: 523659.17307704437
MAE : 223028.11612775121
R2  : 0.8225042348272076


c:\Users\ramir\Documents\GitHub\marzo\ML-idealista\ML-idealista\.venv ML-Idealista\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


### LIGHTGBM
### MAE: 223028.11612775121

 ## CATBOOST:


In [14]:
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_validate

catboost = CatBoostRegressor(random_state=42, verbose=0)

# Catbooost trata las categóricas de forma "nativa" él solo y además gestioina los NaN, por lo que está bien pasarle el df en crudo
# Las extra que nos guardamos para probar con el catboost a propósito
# Lo único que hay que tener en cuenta es que por ello "no sale bien" una pipeline. Ya tiene herramientas él para hacerlo por su cuenta
# Por eso es más tardón, porque hace muchas más cosas
cat_catboost = categoricas + extras

X_cb = train[numericas + cat_catboost + binarias].reset_index(drop=True)

resultado = cross_validate(catboost, X_cb, y_train, cv=cv, params={"cat_features": cat_catboost}, scoring=["neg_root_mean_squared_error", "neg_mean_absolute_error", "r2"])

print("RMSE:", -resultado["test_neg_root_mean_squared_error"].mean())
print("MAE :", -resultado["test_neg_mean_absolute_error"].mean())
print("R2  :", resultado["test_r2"].mean())

RMSE: 500231.8649035301
MAE : 201014.4791617237
R2  : 0.8378295982475924


In [15]:
# Para hacerlo bien bien, hay que tener en cuenta el log del precio, como en el resto de modelos

catboost_log = TransformedTargetRegressor(regressor=CatBoostRegressor(random_state=42, verbose=0), func=np.log1p, inverse_func=np.expm1)

resultado = cross_validate(catboost_log, X_cb, y_train, cv=cv, params={"cat_features": cat_catboost}, scoring=["neg_root_mean_squared_error", "neg_mean_absolute_error", "r2"])

print("MAE :", -resultado["test_neg_mean_absolute_error"].mean())
print("R2  :", resultado["test_r2"].mean())

MAE : 190148.99762636214
R2  : 0.8543198126928052


## OPTIMIZACIÓN: XGBOOST Y CATBOOST
### Optimización bayesiana
### XGBoost + Optuna

In [41]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING) #Para que se calle

# Por recordar, esta es una función que va a testear una y otra vez varios parámetros
# Pero cuando encuentr el rango donde unos se muevan bien, descartará el resto y se acercará a ese rango para ir probando
# Le das el hiperparámetro junto a un min y un max y buscará al principio aleatoriamente hasta dar con ese rango del que hablo
def objective_xgboost(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 200, 2500),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "subsample":        trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10)
        }
    
    model = Pipeline([("prep", preprocessing), ("xgboost", XGBRegressor(**params, random_state=42, n_jobs=-1))])
    model_log = TransformedTargetRegressor(regressor=model, func=np.log1p, inverse_func=np.expm1)

    resultado = cross_validate(model_log, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error")
    return -resultado["test_score"].mean()

# Con el "estudio" le dices qué buscas y llamas a la función con un número de trials, de cantidad de veces que lo va a hacer
study_xgb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgboost, n_trials=50)

print("Mejor MAE:", study_xgb.best_value)
study_xgb.best_params

c:\Users\ramir\Documents\GitHub\marzo\ML-idealista\ML-idealista\.venv ML-Idealista\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Mejor MAE: 219364.809375


{'n_estimators': 1975,
 'learning_rate': 0.013245577617887774,
 'max_depth': 8,
 'subsample': 0.9325894058024048,
 'colsample_bytree': 0.6715508813062192,
 'min_child_weight': 1}

#### CatBoost + Optuna

In [42]:
# Lo mismo que el anterior, pero esto con Catboost
# Es decir, va a tardar más.
def objective_catboost(trial):
    params = {
        "iterations":    trial.suggest_int("iterations", 500, 2500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth":         trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg":   trial.suggest_float("l2_leaf_reg", 1, 10, log=True),
    }
    
    model = CatBoostRegressor(**params, random_state=42, verbose=0)
    model_log = TransformedTargetRegressor(regressor=model, func=np.log1p, inverse_func=np.expm1)

    resultado = cross_validate(model_log, X_cb, y_train, cv=cv, params={"cat_features": cat_catboost}, scoring="neg_mean_absolute_error")
    return -resultado["test_score"].mean()

study_cb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42),
                               storage="sqlite:///optuna_catboost.db",
                               study_name="catboost_madrid",
                               load_if_exists=True)

study_cb.optimize(objective_catboost, n_trials=25)

print("Mejor MAE:", round(study_cb.best_value))
study_cb.best_params

Mejor MAE: 185803


{'iterations': 2044,
 'learning_rate': 0.02820168912904125,
 'depth': 9,
 'l2_leaf_reg': 1.9203884058242628}

# PRUEBA CON OTRAS VARIABLES

In [24]:
# No habíamos incluido TAGS como feature
# Probemos un nuevo modelo con esas tags separadas en etiquetas representativas (que haya en más de un 1% del dataset)

todas_etiquetas = train["tags"].fillna("").str.split(",").explode().str.strip()
todas_etiquetas = todas_etiquetas[todas_etiquetas != ""]
frecuencia = todas_etiquetas.value_counts()
frecuencia

tags
PISO                   4573
VIVIENDA               4286
EXTERIOR               1869
METRO                  1690
AMPLIO                 1524
TERRAZA                1444
OPORTUNIDAD            1399
REFORMADO              1378
EXCLUSIVA              1214
LUMINOSO               1058
HOGAR                   973
INMOBILIARIA            937
ESPECTACULAR            905
ÁTICO                   834
FINCA                   817
VISTAS                  720
LUJO                    715
NUEVO                   714
EQUIPADA                678
EXCLUSIVO               638
REFORMADA               631
PARQUE                  624
CASA                    617
GARAJE                  600
INTERIOR                573
PISCINA                 549
HALL                    509
MODERNO                 497
ESTRENAR                496
APARTAMENTO             491
FUNCIONAL               481
SUITE                   458
ESTUDIO                 410
PRESTIGIOSO             409
AMUEBLADA               408
URBANIZACIÓN   

In [25]:
etiquetas_utiles = frecuencia[frecuencia >= 100].index.tolist()

tags_texto = train["tags"].fillna("")
flags_tags = []

for etiqueta in etiquetas_utiles:
    columna = "tag_" + etiqueta.lower()
    train[columna] = tags_texto.str.contains(etiqueta, regex=False).astype(int)
    flags_tags.append(columna)

train[flags_tags].sum().sort_values(ascending=False)

tag_piso                   4573
tag_vivienda               4286
tag_exterior               1869
tag_metro                  1690
tag_amplio                 1524
tag_terraza                1444
tag_oportunidad            1399
tag_reformado              1378
tag_exclusiva              1214
tag_luminoso               1058
tag_hogar                   973
tag_inmobiliaria            937
tag_espectacular            905
tag_ático                   834
tag_finca                   817
tag_vistas                  720
tag_lujo                    715
tag_nuevo                   714
tag_equipada                678
tag_exclusivo               638
tag_reformada               631
tag_parque                  624
tag_casa                    617
tag_garaje                  600
tag_interior                573
tag_piscina                 549
tag_hall                    509
tag_moderno                 497
tag_estrenar                496
tag_apartamento             491
tag_funcional               481
tag_suit

In [ ]:
import optuna

# Podemos hacer esto gracias a que guardamos en local el estudio de Optuna
# Aunque hayamos cerrado el programa o lo enviemos, ya no tenemos que esperar la hora y media que tardó en sacar
# estos params
# Lo recuperamos del archivo y listo.
study_cb = optuna.load_study(study_name="catboost_madrid", storage="sqlite:///optuna_catboost.db")

print("Mejor MAE:", round(study_cb.best_value))
study_cb.best_params

Mejor MAE: 185803


{'iterations': 2044,
 'learning_rate': 0.02820168912904125,
 'depth': 9,
 'l2_leaf_reg': 1.9203884058242628}

In [ ]:
# Lo que llevamos haciendo todo el notebook, pero con la nueva variable tags y los parámetros que obtuvimos

mejores_params = study_cb.best_params

X_cb_tags = train[numericas + cat_catboost + binarias + flags_tags]

catboost_tags = CatBoostRegressor(**mejores_params, random_state=42, verbose=0)
catboost_tags_log = TransformedTargetRegressor(regressor=catboost_tags, func=np.log1p, inverse_func=np.expm1)

resultado = cross_validate(catboost_tags_log, X_cb_tags, y_train, cv=cv, params={"cat_features": cat_catboost}, scoring=["neg_mean_absolute_error", "r2"])

mae_tags = -resultado["test_neg_mean_absolute_error"].mean()

MAE con tags: 178584
MAE sin tags: 185803
Aportacion : 7219 EUR


In [27]:
print("MAE con tags:", mae_tags)
print("MAE sin tags:", study_cb.best_value)

MAE con tags: 178583.90321007842
MAE sin tags: 185803.1288026239


La mejora es grande. TAGS era una feature importante que no tuvimos tan en cuenta al no haberle hecho un correcto preprocesado.

## Prueba contra test

In [29]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

test = pd.read_csv("../data_sample/test.csv")

# Lo mismo que en el último train sacado al final, con los "tags útiles"
tags_texto_test = test["tags"].fillna("")
for etiqueta in etiquetas_utiles:
    test["tag_" + etiqueta.lower()] = tags_texto_test.str.contains(etiqueta, regex=False).astype(int)

X_test = test[numericas + cat_catboost + binarias + flags_tags]
y_test = test["PrecioActual"]

X_test.shape

(2237, 69)

In [30]:
modelo_final = CatBoostRegressor(**mejores_params, random_state=42, verbose=0)
modelo_final_log = TransformedTargetRegressor(regressor=modelo_final, func=np.log1p, inverse_func=np.expm1)

modelo_final_log.fit(X_cb_tags, y_train, cat_features=cat_catboost)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.","CatBoostRegre...42, verbose=0)"
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,69
regressor_ regressor_: objectFitted regressor.,CatBoostRegressor,"CatBoostRegre...42, verbose=0)"
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,FunctionTransformer,FunctionTrans...validate=True)


In [32]:
pred_test = modelo_final_log.predict(X_test)

mae  = mean_absolute_error(y_test, pred_test)
rmse = np.sqrt(mean_squared_error(y_test, pred_test))
r2   = r2_score(y_test, pred_test)

print("MAE  :", mae)
print("RMSE :", rmse)
print("R2   :", r2)

MAE  : 180709.77338622112
RMSE : 439764.62230136053
R2   : 0.8634756735704264


### FEATURE IMPORTANCE

In [34]:
feature_importance = pd.DataFrame({
    "feature": X_cb_tags.columns,
    "importance": modelo_final_log.regressor_.get_feature_importance(),
})
feature_importance = feature_importance.sort_values("importance", ascending=False)
feature_importance

,feature,importance
0,metros,41.674592
3,zona,23.128974
7,barrio,7.852468
5,ascensor_limpio,4.755588
8,planta_limpio,3.821080
...,...,...
63,tag_solo_particulares,0.028637
46,tag_funcional,0.027419
57,tag_patio,0.026687
64,tag_portero,0.026114


### Terminado el modelo
#### Conclusiones
No se puede afirmar que el modelo sea bueno o malo, sino justo. Aun con un preprocesado intenso, un EDA perfectamente ejecutado y mucho trabajo detrás del modelo, el DataSet da para lo que da.  
Con sus 16 variables iniciales, es aún insuficiente para "predecir" un verdadero piso. Además, la métrica global está inflada por la cola de viviendas de lujo o de barrios que destacan sobre otros, por lo que no describe en su totalidad la ciudad de Madrid.  
No podemos entrenar dos modelos distintos, así como tampoco prescindir de las viviendas de lujo, pues son una parte importante de este dataset.  
Es por eso que el modelo funciona de esta manera. Es justo con lo que tenemos.

## Guardar el modelo

In [40]:
import joblib

joblib.dump(modelo_final_log, "../models/catboost_madrid.pkl")

['../models/catboost_madrid.pkl']